# 02 · Pelatihan U-Net dan DeepLabV3+ (T4-A, Raabin-WBC)

**Cara pakai (tiap anggota, di akun Colab masing-masing)**
1. Ketua sudah menaruh `data.py`, `runlog.py` (dan folder `splits/`) di folder Drive bersama. Taruh `metrics.py`, `models.py`, `train.py` di folder `code/` yang sama.
2. Runtime → Change runtime type → **T4 GPU**.
3. Isi `OWNER` di sel setup dengan **satu nama tetap** (mis. nama depan). Nama ini menentukan file log `logs/log_<OWNER>.csv`.
4. Jalankan sel setup, lalu **sel uji cepat**. Setelah itu jalankan hanya sel milikmu.

**Aturan**
- Satu faktor berubah per perbandingan, sisanya tetap pada baseline (U-Net ResNet-34, CE+Dice, split `grouped`, augmentasi warna aktif, 256 px, seed 42).
- Semua pemilihan epoch memakai **validasi**. `eval_test=True` hanya untuk konfigurasi final, sekali, di akhir.
- Jangan mengubah file di `splits/`.
- Run yang gagal tetap tercatat otomatis. Run yang terputus karena sesi Colab habis TIDAK tercatat, catat manual.

In [2]:
# ===== SEL SETUP (jalankan tiap sesi Colab baru) =====
from google.colab import drive; drive.mount('/content/drive')
!apt-get install -y unrar > /dev/null 2>&1
!pip -q install albumentations segmentation-models-pytorch
DRIVE = "/content/drive/MyDrive/PROYEK UAS DEEP LEARNING"

!mkdir -p /content/raabin_gt/bmp
!unrar x -inul "{DRIVE}/GroundTruths_bmpformat.rar" /content/raabin_gt/bmp/

import sys; sys.path.append(f"{DRIVE}/code")
import warnings; warnings.filterwarnings("ignore")
from train import run_experiment, run_seeds

OWNER = "Adit"      # satu nama tetap, tanpa spasi
assert OWNER and " " not in OWNER, "OWNER tidak boleh kosong atau mengandung spasi"
print("OWNER =", OWNER)

Mounted at /content/drive
OWNER = Adit


## Uji cepat (1 epoch)
Memastikan GPU, unduhan bobot pralatih, dan penulisan log berjalan. Tercatat di log dengan `ablation='smoke_test'`; baris ini dibuang saat laporan. Perkiraan waktu 1-3 menit (belum diukur di T4).

In [2]:
run_experiment(OWNER, "smoke_test", "-", epochs=1)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

ep  1/1 | loss 0.7146 | val dice cyto 0.7601 nuc 0.9087 | skor 0.8344


{'val_dice_bg': 0.9848,
 'val_iou_bg': 0.9707,
 'val_dice_cyto': 0.7601,
 'val_dice_cyto_sd': 0.117,
 'val_iou_cyto': 0.6264,
 'val_dice_nuc': 0.9087,
 'val_dice_nuc_sd': 0.0699,
 'val_iou_nuc': 0.8396,
 'val_miou': 0.8122,
 'val_miou_sd': 0.0683,
 'val_hd95_cyto': 23.0459,
 'val_hd95_cyto_sd': 20.8442,
 'val_hd95_nuc': 16.0177,
 'val_hd95_nuc_sd': 16.2763,
 'run_id': 'Adit_1123_070244',
 'best_epoch': 1}

## Baseline (dijalankan SEKALI untuk seluruh tim)
Baseline U-Net milik **Anggota 2**, baseline DeepLabV3+ milik **Anggota 3**. Semua ablasi dibandingkan dengan baseline U-Net ini. Jangan dijalankan ulang oleh anggota lain.

In [3]:
# Baseline U-Net (Anggota 2)
run_experiment(OWNER, "baseline", "unet_ce_dice", arch="unet")

ep  1/20 | loss 0.6714 | val dice cyto 0.8027 nuc 0.9397 | skor 0.8712
ep  2/20 | loss 0.2167 | val dice cyto 0.8509 nuc 0.9658 | skor 0.9083
ep  3/20 | loss 0.1688 | val dice cyto 0.7965 nuc 0.8546 | skor 0.8256
ep  4/20 | loss 0.1538 | val dice cyto 0.8637 nuc 0.9674 | skor 0.9156
ep  5/20 | loss 0.1344 | val dice cyto 0.8480 nuc 0.9652 | skor 0.9066
ep  6/20 | loss 0.1366 | val dice cyto 0.8433 nuc 0.9584 | skor 0.9009
ep  7/20 | loss 0.1304 | val dice cyto 0.8848 nuc 0.9714 | skor 0.9281
ep  8/20 | loss 0.1243 | val dice cyto 0.7954 nuc 0.9593 | skor 0.8774
ep  9/20 | loss 0.1160 | val dice cyto 0.8800 nuc 0.9698 | skor 0.9249
ep 10/20 | loss 0.1175 | val dice cyto 0.8837 nuc 0.9693 | skor 0.9265
ep 11/20 | loss 0.1074 | val dice cyto 0.8842 nuc 0.9646 | skor 0.9244
ep 12/20 | loss 0.1073 | val dice cyto 0.8877 nuc 0.9724 | skor 0.9301
ep 13/20 | loss 0.1023 | val dice cyto 0.8870 nuc 0.9713 | skor 0.9292
ep 14/20 | loss 0.1015 | val dice cyto 0.8872 nuc 0.9690 | skor 0.9281
ep 15/

{'val_dice_bg': 0.9946,
 'val_iou_bg': 0.9894,
 'val_dice_cyto': 0.8963,
 'val_dice_cyto_sd': 0.0833,
 'val_iou_cyto': 0.8206,
 'val_dice_nuc': 0.9724,
 'val_dice_nuc_sd': 0.0103,
 'val_iou_nuc': 0.9465,
 'val_miou': 0.9188,
 'val_miou_sd': 0.0391,
 'val_hd95_cyto': 8.5563,
 'val_hd95_cyto_sd': 14.0277,
 'val_hd95_nuc': 4.0938,
 'val_hd95_nuc_sd': 1.3113,
 'run_id': 'Adit_1123_070320',
 'best_epoch': 17}

In [ ]:
# Baseline DeepLabV3+ (Anggota 3)
run_experiment(OWNER, "baseline", "deeplabv3plus_ce_dice", arch="deeplabv3plus")

## Ablasi per anggota (U-Net, 1 seed = 42, evaluasi di VALIDASI)
Nilai yang sama dengan baseline tidak dijalankan lagi. Jika memori GPU habis (terutama 384 px), turunkan `batch_size` ke 8 dan itu tercatat di log.

In [ ]:
# Anggota 1 (Ketua): skema split. Baseline memakai 'grouped'; ini menambah 'random' (bocor antar-apusan)
run_experiment(OWNER, "split", "random", split="random")

In [4]:
# Anggota 2: fungsi loss. Baseline = ce_dice; tambahkan tiga lainnya
for l in ["ce", "dice", "focal_dice"]:
    run_experiment(OWNER, "loss", l, loss=l)

ep  1/20 | loss 0.2976 | val dice cyto 0.8067 nuc 0.9259 | skor 0.8663
ep  2/20 | loss 0.0823 | val dice cyto 0.7465 nuc 0.9357 | skor 0.8411
ep  3/20 | loss 0.0634 | val dice cyto 0.8703 nuc 0.9659 | skor 0.9181
ep  4/20 | loss 0.0555 | val dice cyto 0.8034 nuc 0.9340 | skor 0.8687
ep  5/20 | loss 0.0560 | val dice cyto 0.8563 nuc 0.9678 | skor 0.9121
ep  6/20 | loss 0.0519 | val dice cyto 0.8607 nuc 0.9605 | skor 0.9106
ep  7/20 | loss 0.0465 | val dice cyto 0.8672 nuc 0.9645 | skor 0.9159
ep  8/20 | loss 0.0414 | val dice cyto 0.8823 nuc 0.9683 | skor 0.9253
ep  9/20 | loss 0.0390 | val dice cyto 0.8871 nuc 0.9704 | skor 0.9287
ep 10/20 | loss 0.0394 | val dice cyto 0.8873 nuc 0.9693 | skor 0.9283
ep 11/20 | loss 0.0371 | val dice cyto 0.8843 nuc 0.9671 | skor 0.9257
ep 12/20 | loss 0.0368 | val dice cyto 0.8845 nuc 0.9690 | skor 0.9267
ep 13/20 | loss 0.0358 | val dice cyto 0.8840 nuc 0.9692 | skor 0.9266
ep 14/20 | loss 0.0345 | val dice cyto 0.8803 nuc 0.9662 | skor 0.9232
early 

In [5]:
# Anggota 2
run_experiment(OWNER, "loss", "ce_p20", loss="ce", patience=20)

ep  1/20 | loss 0.3002 | val dice cyto 0.6703 nuc 0.8526 | skor 0.7614
ep  2/20 | loss 0.0903 | val dice cyto 0.8152 nuc 0.9665 | skor 0.8909
ep  3/20 | loss 0.0656 | val dice cyto 0.8053 nuc 0.9561 | skor 0.8807
ep  4/20 | loss 0.0547 | val dice cyto 0.8572 nuc 0.9684 | skor 0.9128
ep  5/20 | loss 0.0500 | val dice cyto 0.8611 nuc 0.9672 | skor 0.9142
ep  6/20 | loss 0.0458 | val dice cyto 0.8618 nuc 0.9654 | skor 0.9136
ep  7/20 | loss 0.0432 | val dice cyto 0.8604 nuc 0.9684 | skor 0.9144
ep  8/20 | loss 0.0416 | val dice cyto 0.8880 nuc 0.9690 | skor 0.9285
ep  9/20 | loss 0.0412 | val dice cyto 0.8654 nuc 0.9624 | skor 0.9139
ep 10/20 | loss 0.0408 | val dice cyto 0.8804 nuc 0.9684 | skor 0.9244
ep 11/20 | loss 0.0377 | val dice cyto 0.8658 nuc 0.9515 | skor 0.9086
ep 12/20 | loss 0.0361 | val dice cyto 0.8837 nuc 0.9703 | skor 0.9270
ep 13/20 | loss 0.0355 | val dice cyto 0.8881 nuc 0.9702 | skor 0.9291
ep 14/20 | loss 0.0350 | val dice cyto 0.8803 nuc 0.9655 | skor 0.9229
ep 15/

{'val_dice_bg': 0.9946,
 'val_iou_bg': 0.9894,
 'val_dice_cyto': 0.8953,
 'val_dice_cyto_sd': 0.0877,
 'val_iou_cyto': 0.8198,
 'val_dice_nuc': 0.9719,
 'val_dice_nuc_sd': 0.0117,
 'val_iou_nuc': 0.9455,
 'val_miou': 0.9182,
 'val_miou_sd': 0.0407,
 'val_hd95_cyto': 8.739,
 'val_hd95_cyto_sd': 15.394,
 'val_hd95_nuc': 4.4148,
 'val_hd95_nuc_sd': 3.457,
 'run_id': 'Adit_1123_073423',
 'best_epoch': 17}

In [ ]:
# Anggota 3: augmentasi warna. Baseline = aktif; ini menambah 'nonaktif'
run_experiment(OWNER, "color_aug", "off", color_aug=False)

In [ ]:
# Anggota 4: resolusi masukan. Baseline = 256; ini menambah 384
run_experiment(OWNER, "resolution", "384", img_size=384)

In [ ]:
# Opsional bila waktu cukup: encoder pralatih vs inisialisasi acak
run_experiment(OWNER, "pretrained", "random_init", pretrained=False)

## Perbandingan utama: 3 seed untuk dua konfigurasi terbaik
Pilih dua konfigurasi berdasarkan hasil **validasi** (misalnya U-Net vs DeepLabV3+ dengan pengaturan terbaik). `eval_test=True` menyentuh set uji, jadi jalankan ini **hanya setelah semua ablasi selesai dan konfigurasinya final**, sekali saja. `save_path` menyimpan bobot untuk model card (satu file ~100 MB).

In [6]:
#anggota 2
import pandas as pd
res = []
for s in (42, 43, 44):
    res.append(run_experiment(OWNER, "main", "unet_final", arch="unet", seed=s,
                              eval_test=True,
                              save_path=f"{DRIVE}/checkpoints/unet_final_seed{s}.pt"))
df_unet = pd.DataFrame(res)
cols = [c for c in df_unet.columns if c.startswith(("val_", "test_")) and not c.endswith("_sd")]
print(pd.DataFrame({"mean": df_unet[cols].mean(), "sd": df_unet[cols].std(ddof=1)}).round(4))

ep  1/20 | loss 0.7179 | val dice cyto 0.7385 nuc 0.9661 | skor 0.8523
ep  2/20 | loss 0.2147 | val dice cyto 0.8201 nuc 0.9450 | skor 0.8825
ep  3/20 | loss 0.1729 | val dice cyto 0.8081 nuc 0.8886 | skor 0.8483
ep  4/20 | loss 0.1557 | val dice cyto 0.7948 nuc 0.9576 | skor 0.8762
ep  5/20 | loss 0.1386 | val dice cyto 0.8700 nuc 0.9649 | skor 0.9174
ep  6/20 | loss 0.1355 | val dice cyto 0.8370 nuc 0.9569 | skor 0.8969
ep  7/20 | loss 0.1270 | val dice cyto 0.8675 nuc 0.9646 | skor 0.9161
ep  8/20 | loss 0.1203 | val dice cyto 0.8778 nuc 0.9662 | skor 0.9220
ep  9/20 | loss 0.1147 | val dice cyto 0.8579 nuc 0.9673 | skor 0.9126
ep 10/20 | loss 0.1222 | val dice cyto 0.8844 nuc 0.9681 | skor 0.9263
ep 11/20 | loss 0.1079 | val dice cyto 0.8866 nuc 0.9660 | skor 0.9263
ep 12/20 | loss 0.1091 | val dice cyto 0.8728 nuc 0.9650 | skor 0.9189
ep 13/20 | loss 0.1048 | val dice cyto 0.8845 nuc 0.9718 | skor 0.9282
ep 14/20 | loss 0.1010 | val dice cyto 0.8908 nuc 0.9711 | skor 0.9309
ep 15/

In [3]:
for s in (42, 43, 44):
    run_experiment(OWNER, "cross_session", "unet_cs_p20", arch="unet", seed=s,
                   split="cross_session", eval_test=True, patience=20)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

ep  1/20 | loss 0.8874 | val dice cyto 0.6610 nuc 0.8776 | skor 0.7693
ep  2/20 | loss 0.2452 | val dice cyto 0.8023 nuc 0.9560 | skor 0.8791
ep  3/20 | loss 0.1839 | val dice cyto 0.8608 nuc 0.9655 | skor 0.9132
ep  4/20 | loss 0.1637 | val dice cyto 0.8571 nuc 0.9612 | skor 0.9092
ep  5/20 | loss 0.1547 | val dice cyto 0.8366 nuc 0.9435 | skor 0.8901
ep  6/20 | loss 0.1433 | val dice cyto 0.8438 nuc 0.9386 | skor 0.8912
ep  7/20 | loss 0.1249 | val dice cyto 0.8552 nuc 0.9495 | skor 0.9023
ep  8/20 | loss 0.1246 | val dice cyto 0.8770 nuc 0.9605 | skor 0.9187
ep  9/20 | loss 0.1154 | val dice cyto 0.8807 nuc 0.9543 | skor 0.9175
ep 10/20 | loss 0.1140 | val dice cyto 0.8889 nuc 0.9683 | skor 0.9286
ep 11/20 | loss 0.1176 | val dice cyto 0.8884 nuc 0.9684 | skor 0.9284
ep 12/20 | loss 0.1141 | val dice cyto 0.8865 nuc 0.9616 | skor 0.9241
ep 13/20 | loss 0.1134 | val dice cyto 0.9004 nuc 0.9691 | skor 0.9347
ep 14/20 | loss 0.1036 | val dice cyto 0.9032 nuc 0.9697 | skor 0.9365
ep 15/

In [ ]:
df_dlv3, tab_dlv3 = run_seeds(owner=OWNER, ablation="main", factor_value="deeplabv3plus_final",
                              arch="deeplabv3plus", eval_test=True,
                              save_path=f"{DRIVE}/checkpoints/deeplabv3plus_final.pt")

## Gabung seluruh log (Ketua, di akhir)
Menghasilkan tabel log eksperimen, total jam GPU, dan bukti bahwa set uji hanya dipakai pada run final.

In [ ]:
import glob, pandas as pd
log = pd.concat([pd.read_csv(f) for f in glob.glob(f"{DRIVE}/logs/log_*.csv")], ignore_index=True)
log = log[log.ablation != "smoke_test"]
log.to_csv(f"{DRIVE}/logs/LOG_GABUNGAN.csv", index=False)
print("Total run:", len(log), "| gagal:", (log.status == "failed").sum())
print("Total jam GPU:", round(log.gpu_minutes.sum() / 60, 2))
print("Run yang memakai set uji:", int(log.used_test.sum()))
print(log.groupby("owner").gpu_minutes.sum().div(60).round(2))